In [1]:
import numpy as np  # import numerical python
import matplotlib.pyplot as plt  # import plotting functions
import seaborn as sns  # import nicer plotting functions
import pandas as pd
import polars as pl
import tifffile as tiff
from tifffile import imwrite, imread
from copy import deepcopy
import os
import time

import sys

sys.path.append("../..")

from src import IOFunctions

IO = IOFunctions.IO_Functions()

from src import PlottingFunctions

plotter = PlottingFunctions.Plotter(dark_background=True)

from src import ImageAnalysisFunctions

I_AF = ImageAnalysisFunctions.Image_Analysis_Functions()

from src import sCMOSFunctions

sCMOS = sCMOSFunctions.sCMOS_Functions()

from src import PSFFunctions

PSF = PSFFunctions.PSF_Functions()

from src import SpectralFunctions

S_F = SpectralFunctions.Spectral_Funcs()

from src import MaskFunctions

M_F = MaskFunctions.Mask_Functions()

from src import SpotDetectionFunctions

SD_F = SpotDetectionFunctions.SpotDetection_Functions()

from src import SR_Functions

SRes_F = SR_Functions.SuperRes_Functions()

from src import HelperFunctions

H_F = HelperFunctions.Helper_Functions()

from src import SM_extractionfunctions

SM_E = SM_extractionfunctions.extract_SMs()

from src.DriftCorrectionFunctions import (
    Drift_Correction_Functions,
    DriftMethod,
    DriftParameters,
    DriftResult,
)

import types

smoothing_function = types.SimpleNamespace()
smoothing_function.args = {"sigma": 1.5}
smoothing_function.extent = 1.5
smoothing_function.smoothing_function = sCMOS.gaussian_filter_stack
smoothing_function.data_arg = "image"

/tmp/ipykernel_1527855/3263995668.py:22: DeprecationWarning: PlottingFunctions.Plotter is deprecated and will be removed in a future version. Use PlottingBase.PublicationPlotter directly instead.
  plotter = PlottingFunctions.Plotter(dark_background=True)


In [2]:
data_folder = "../../Camera_Calibrations/Ximea_Camera/"
gain = IO.read_tiff(os.path.join(data_folder, "gain.tif"))
offset = IO.read_tiff(os.path.join(data_folder, "offset.tif"))
variance = IO.read_tiff(os.path.join(data_folder, "variance.tif"))
readnoise = IO.read_tiff(os.path.join(data_folder, "readnoise.tif"))
rqe = IO.read_tiff(os.path.join(data_folder, "rqe.tif"))
R, G, B, wavelength = S_F.getpixelefficiency()

In [3]:
objective_T = S_F.getobjectiveefficiency(wavelength)

In [4]:
wavelength = wavelength
pixel_QYs = np.vstack([B, G, R])

In [18]:
pd.read_hdf(localisation_files[0])['chi_sqr']

0        1.260966
1        1.133988
2        1.067377
3        0.968019
4        1.136072
           ...   
24494    1.118721
24495    1.321068
24496    0.969399
24497    0.948258
24498    1.193222
Name: chi_sqr, Length: 24499, dtype: float32

In [19]:
folders = '/scratch/sycamore-asap/2026_Multicolour_Paper/Data/Ximea/Single_Dye_Experiments'
folder_in_folder = os.listdir(folders)

min_cluster_size = 10
chi_val = 2
max_localisation_error = 1.0
min_photons = 500
max_photons = 1e6
max_distance = 0.5

for folder in folder_in_folder:
    example_folder = os.path.join(folders, folder)
    localisation_files = H_F.file_search(example_folder, ".h5", "Pos")
    localisation_files = np.array([x for x in localisation_files if 'database' not in x])
    single_molecule_database, single_frame_database = SM_E.extract_single_molecules_DBSCAN(
        localisation_files,
        chi_val=chi_val,
        min_cluster_size=min_cluster_size,
        max_localisation_error=max_localisation_error,
        min_photons=min_photons,
        max_photons=max_photons,
    )

[ '/scratch/sycamore-asap/2026_Multicolour_Paper/Data/Ximea/Single_Dye_Experiments/ATTO700/40mW638_both_NF_785sp_1/40mW638_both_NF_785sp_1_MMStack_3-Pos000_000.h5'
 '/scratch/sycamore-asap/2026_Multicolour_Paper/Data/Ximea/Single_Dye_Experiments/ATTO700/40mW638_both_NF_785sp_1/40mW638_both_NF_785sp_1_MMStack_3-Pos000_001.h5'
 '/scratch/sycamore-asap/2026_Multicolour_Paper/Data/Ximea/Single_Dye_Experiments/ATTO700/40mW638_both_NF_785sp_1/40mW638_both_NF_785sp_1_MMStack_3-Pos000_002.h5'
 '/scratch/sycamore-asap/2026_Multicolour_Paper/Data/Ximea/Single_Dye_Experiments/ATTO700/40mW638_both_NF_785sp_1/40mW638_both_NF_785sp_1_MMStack_3-Pos000_003.h5'
 '/scratch/sycamore-asap/2026_Multicolour_Paper/Data/Ximea/Single_Dye_Experiments/ATTO700/40mW638_both_NF_785sp_1/40mW638_both_NF_785sp_1_MMStack_3-Pos001_000.h5'
 '/scratch/sycamore-asap/2026_Multicolour_Paper/Data/Ximea/Single_Dye_Experiments/ATTO700/40mW638_both_NF_785sp_1/40mW638_both_NF_785sp_1_MMStack_3-Pos001_001.h5'
 '/scratch/sycamore-a

IndexError: only integers, slices (`:`), ellipsis (`...`), numpy.newaxis (`None`) and integer or boolean arrays are valid indices

# this is an example of linking molecules with HDBSCAN ###

In [ ]:
single_molecule_database, single_frame_database = SM_E.extract_single_molecules_HDBSCAN(
    localisation_files,
    min_cluster_size=min_cluster_size,
    max_localization_error=max_localization_error,
    min_photons=min_photons,
    max_photons=max_photons,
)

# this is an example of linking molecules with DBSCAN

In [ ]:
single_molecule_database, single_frame_database = SM_E.extract_single_molecules_DBSCAN(
    localisation_files,
    min_cluster_size=min_cluster_size,
    max_localization_error=max_localization_error,
    min_photons=min_photons,
    max_photons=max_photons,
)

# this is an example of linking molecules sequentially

In [ ]:
extract_single_molecules_linked = SM_E.extract_single_molecules_linked(
    localisation_files,
    max_distance=max_distance,
    max_localization_error=max_localization_error,
    min_photons=min_photons,
    max_photons=max_photons,
)